Installations

In [1]:
!pip install -U langchain langchain-groq langchain-community \
    langchain-text-splitters langchain-huggingface \
    sentence-transformers faiss-cpu pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 38.4 MB/s eta 0:00:00
  Attempting uninstall: faiss-cpu
    Found existing installation: faiss-cpu 1.15.0
    Uninstalling faiss-cpu-1.15.0:
      Successfully uninstalled faiss-cpu-1.15.0


Imports

In [2]:
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_groq import ChatGroq
from getpass import getpass

/tmp/ipykernel_18372/3105367800.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDFs path for RAG

In [3]:
pdf_paths = [
    "data/interview_tips.pdf",
    "data/cv_writing_tips.pdf",
    "data/my_cv.pdf"
]

Read PDFs

In [4]:
pdf_docs = []

for path in pdf_paths:
    loader = PyPDFLoader(path)
    pdf_docs.extend(loader.load())

print(f"Loaded {len(pdf_docs)} pages from PDFs")

Loaded 26 pages from PDFs


Read Jobs file

In [5]:
df = pd.read_csv("data/jobs.csv")

print(df.head())
print(f"\nLoaded {len(df)} job entries")

                           Title  \
0                   Data Analyst   
1        Machine Learning Intern   
2                 Data Scientist   
3                    AI Engineer   
4  Business Intelligence Analyst   

                                           Skills Experience Location  
0                    Python, SQL, Excel, Power BI  0-2 years    Cairo  
1  Python, Scikit-learn, Pandas, Machine Learning  0-1 years    Cairo  
2       Python, SQL, Machine Learning, Statistics  1-3 years    Cairo  
3      Python, PyTorch, TensorFlow, Deep Learning  0-2 years    Cairo  
4        Power BI, SQL, Excel, Data Visualization  0-2 years    Cairo  

Loaded 5 job entries


Convert every job to text (Because RAG needs text not rows)

In [6]:
job_texts = []

for _, row in df.iterrows():
    text = (
        f"Job Title: {row['Title']}. "
        f"Required Skills: {row['Skills']}. "
        f"Experience: {row['Experience']}. "
        f"Location: {row['Location']}."
    )
    job_texts.append(text)

print(f"Loaded {len(job_texts)} job entries")

Loaded 5 job entries


Chunking PDFs using recursive chunking for preserving the meaning and adding Metadata



In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

pdf_chunks = splitter.split_documents(pdf_docs)

# Add source type metadata
for doc in pdf_chunks:
    source = doc.metadata.get("source", "").lower()

    if "my_cv" in source:
        doc.metadata["source_type"] = "cv"
    elif "cv_writing_tips" in source:
        doc.metadata["source_type"] = "cv_guide"
    elif "interview_tips" in source:
        doc.metadata["source_type"] = "interview_guide"

print(f"Created {len(pdf_chunks)} PDF chunks")

# Debug: تأكيد إن كل نوع اتصنف صح
from collections import Counter
type_counts = Counter(doc.metadata.get("source_type", "UNSET") for doc in pdf_chunks)
print("Chunk counts per source_type:", dict(type_counts))

Created 153 PDF chunks
Chunk counts per source_type: {'interview_guide': 66, 'cv_guide': 72, 'cv': 15}


Converting job data into Documents to combine with PDF documents before creating the vector store.

In [8]:
job_chunks = [
    Document(
        page_content=text,
        metadata={"source_type": "jobs"}
    )
    for text in job_texts
]

print(f"Created {len(job_chunks)} job documents")

Created 5 job documents


Add all chunks

In [9]:
all_chunks = pdf_chunks + job_chunks

print(f"Total documents/chunks: {len(all_chunks)}")

Total documents/chunks: 158


Create Embeddings using all-MiniLM-L6-v2 model

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded


Build FAISS Vector Store for embedding storage

In [11]:
vectorstore = FAISS.from_documents(
    all_chunks,
    embeddings
)

print("Vector store built successfully")

Vector store built successfully


Retriever get best 3 chunks that answer question and send it to LLM

In [12]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("General retriever ready")

General retriever ready


Build Source-Specific FAISS

In [13]:
cv_vectorstore = FAISS.from_documents(
    [doc for doc in all_chunks if doc.metadata.get("source_type") == "cv"],
    embeddings
)

jobs_vectorstore = FAISS.from_documents(
    [doc for doc in all_chunks if doc.metadata.get("source_type") == "jobs"],
    embeddings
)

cv_guide_vectorstore = FAISS.from_documents(
    [doc for doc in all_chunks if doc.metadata.get("source_type") == "cv_guide"],
    embeddings
)

interview_vectorstore = FAISS.from_documents(
    [doc for doc in all_chunks if doc.metadata.get("source_type") == "interview_guide"],
    embeddings
)

print("Source-specific vector stores created successfully")

Source-specific vector stores created successfully


Source-Specific Retrievers

In [14]:
cv_retriever = cv_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

jobs_retriever = jobs_vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

cv_guide_retriever = cv_guide_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

interview_retriever = interview_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Source-specific retrievers ready")

Source-specific retrievers ready


Retriever Routing

In [15]:
def retrieve_documents(question):

    question_lower = question.lower()

    # Handle numbered job questions
    job_numbers = {
        "first": 0,
        "second": 1,
        "third": 2,
        "fourth": 3,
        "fifth": 4
    }

    for word, index in job_numbers.items():
        if f"{word} job" in question_lower:

            job_docs = []

            for i, (_, row) in enumerate(df.iterrows()):
                job_docs.append(
                    Document(
                        page_content=(
                            f"Job {i + 1}: "
                            f"Job Title: {row['Title']}. "
                            f"Required Skills: {row['Skills']}. "
                            f"Experience: {row['Experience']}. "
                            f"Location: {row['Location']}."
                        ),
                        metadata={
                            "source_type": "jobs",
                            "job_number": i + 1
                        }
                    )
                )

            return job_docs

    # Questions that require BOTH CV and Jobs
    if (
        ("job" in question_lower or "jobs" in question_lower)
        and ("my cv" in question_lower or "my resume" in question_lower)
    ):
        cv_docs = cv_retriever.invoke(question)
        job_docs = jobs_retriever.invoke(question)

        return cv_docs + job_docs

    # CV questions
    elif "my cv" in question_lower or "my resume" in question_lower:
        return cv_retriever.invoke(question)

    # Job questions
    elif "job" in question_lower or "jobs" in question_lower:
        return jobs_retriever.invoke(question)

    # Interview questions
    elif "interview" in question_lower:
        return interview_retriever.invoke(question)

    # CV tips
    elif "cv tips" in question_lower or "resume tips" in question_lower:
        return cv_guide_retriever.invoke(question)

    # General questions
    else:
        return retriever.invoke(question)

Test Retriever

In [16]:
question = "What skills are mentioned in my CV?"

docs = retrieve_documents(question)
print(f"Retrieved {len(docs)} documents\n")

for i, doc in enumerate(docs):
    print(f"--- Document {i+1} ---")
    print("Source:", doc.metadata.get("source_type"))
    print(doc.page_content[:300])
    print()

Retrieved 3 documents

--- Document 1 ---
Source: cv
EDUCATION
Helwan University – Faculty of Science
Bachelor / Computer Science and Statistics
2023 – 2027
CGPA: 3.526
SKILLS
Python
NumPy
Microsoft Power BI
Hyperparameter Tuning
Deep Learning
RAG-Systems
Model Evaluation
OpenCV
Problem Solving
SQL
Pandas
Microsoft Excel
NLP
Agentic Ai
LLMs
TensorFlow

--- Document 2 ---
Source: cv
Learning workflows.
02/2026 – 09/2026
•Built ML and Deep Learning solutions using Scikit-learn, TensorFlow, YOLO, 
Transformers, and Hugging Face.
•Developed NLP, Generative AI, and deployment projects using Streamlit, FastAPI, 
and Docker.
PROJECTS
Student Stress Level Prediction | Analysis, EDA, C

--- Document 3 ---
Source: cv
distributions, correlations, data quality, and outliers to identify key factors related to stress levels.
•Built and evaluated multiple classification models, including Logistic Regression, SVM, Random Forest, and 
Naive Bayes, using feature scaling and performance metrics such as a

Calling Groq LLM

In [17]:
groq_api_key = getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
    temperature=0
)

print("LLM initialized successfully")

Enter your Groq API key: ··········
LLM initialized successfully


Prompt

In [18]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful Career Assistant.

Use the provided context and conversation history to answer the user's question.

IMPORTANT RULES:
- Use ONLY the information provided in the context and conversation history.
- Do not invent or assume information.
- If information is not available, clearly say that you do not have enough information.
- The current Context is the primary source of truth for the current question.
- Conversation history should only be used to understand references such as "it", "that job", or "the second job".
- If the current Context explicitly identifies a numbered job, ALWAYS use that job and do not replace it with a job mentioned earlier in the conversation.
- When the user says "first job", "second job", "third job", etc., the numbered job in the current Context is authoritative.
- When comparing the CV with jobs, identify matching skills and missing or unclear requirements.
- Do not invent salary, experience, skills, or other job details that are not provided.

Conversation history:
{chat_history}

Current Context:
{context}
"""
    ),
    ("human", "{input}")
])

print("Prompt updated successfully")

Prompt updated successfully


Conversation Memory

In [19]:
chat_history = []

def format_history(history, max_turns=5):
    if not history:
        return "No previous conversation."
    recent = history[-max_turns:]
    return "\n".join([f"User: {q}\nAssistant: {a}" for q, a in recent])

print("Memory initialized")

Memory initialized


Connect Retriever to LLM

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": RunnableLambda(
            lambda q: format_docs(
                retrieve_documents(q)
            )
        ),
        "chat_history": RunnableLambda(
            lambda q: format_history(chat_history)
        ),
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

Ask Function

In [21]:
def ask(question):
    response = rag_chain.invoke(question)
    chat_history.append((question, response.content))
    return response.content

Testing

In [22]:
print("Q1:", "What skills are mentioned in my CV?")
print("A1:", ask("What skills are mentioned in my CV?"))

print()

print("Q2:", "Which job from the list fits me best based on my CV?")
print("A2:", ask("Which job from the list fits me best based on my CV?"))

print()

print("Q3:", "And what about the second job on the list?")
print("A3:", ask("And what about the second job on the list?"))

print()

print("Q4:", "Does it require SQL?")
print("A4:", ask("Does it require SQL?"))

Q1: What skills are mentioned in my CV?
A1: Here are all the skills and technical tools that appear in the CV you provided:

**Core Skills (listed in the “SKILLS” section)**  
- Python  
- NumPy  
- Microsoft Power BI  
- Hyper‑parameter Tuning  
- Deep Learning  
- Retrieval‑Augmented Generation (RAG) Systems  
- Model Evaluation  
- OpenCV  
- Problem Solving  
- SQL  
- Pandas  
- Microsoft Excel  
- Natural Language Processing (NLP)  
- Agentic AI  
- Large Language Models (LLMs)  
- TensorFlow  
- Streamlit  
- Critical Thinking  
- C++  
- Data Visualization  
- Machine Learning  
- Computer Vision  
- Fine‑Tuning  
- LangChain  
- FastAPI  
- Teamwork  
- Working Under Pressure  

**Additional tools/technologies mentioned elsewhere in the CV**  
- Scikit‑learn  
- YOLO (You Only Look Once)  
- Transformers (model architecture)  
- Hugging Face libraries  
- Docker  

These constitute the full set of skills and technical competencies referenced in your CV.

Q2: Which job from the

In [23]:
print("Q5:", "What are some important CV writing tips?")
print("A5:", ask("What are some important CV writing tips?"))

print()

print("Q6:", "What are some important interview tips?")
print("A6:", ask("What are some important interview tips?"))

print()

print("Q7:", "What is the salary of the AI Engineer job?")
print("A7:", ask("What is the salary of the AI Engineer job?"))

Q5: What are some important CV writing tips?
A5: Here are the key CV‑writing tips that are highlighted in the guide you provided:

| Tip | Why it matters (as described in the guide) |
|-----|--------------------------------------------|
| **Start each bullet with a strong action verb** | Shows you actually did something rather than just describing a duty. |
| **Give the context for the action** | Use quantitative or qualitative details so the reader understands the situation you were working in. |
| **End with the result or impact** | Demonstrates the value you added (e.g., saved time, increased revenue, improved quality). |
| **Use present tense for current roles and past tense for former roles** | Keeps the description consistent and clear about what you’re doing now versus what you did before. |
| **Keep margins between 0.75” and 1” (no less than 0.5”)** | Ensures the document looks tidy and is easy to read. |
| **Use a consistent font style and size (10‑12 pt)** | Gives a professio

Clear History

In [24]:
chat_history = []

Some Test Cases

CV pdf Test

In [25]:
print(ask("What are some important CV writing tips?"))

Here are the key tips highlighted in the guide:

- **Start each bullet with a strong action verb** – this shows you did something concrete.  
- **Provide context** for the action, using quantitative (numbers, percentages) or qualitative details to explain the situation.  
- **Show the end result** so the reader can see the value of your contribution.  
- **Tense usage:**  
  - Use **present tense** for your current role.  
  - Use **past tense** for previous positions.  
- **Formatting basics:**  
  - Set margins between **0.75” and 1”** (no less than 0.5”).  
  - Use a **consistent font style and size** (10‑12 pt).  
  - Leave off your mailing address; **phone number and email are sufficient**.  
- **Structure:**  
  - List each section’s content in **reverse‑chronological order** (most recent first).  
- **Readability:**  
  - Design the resume so it can be **scanned in 15‑30 seconds** – keep it clean, well‑spaced, and easy to digest.  
- **Iterative process:**  
  - Recognize that r

Interview pdf test

In [26]:
print(ask("What are some important interview tips?"))

Here are the key interview‑preparation tips highlighted in the Interview Guide you provided:

| Tip | Why it matters | How to apply it |
|-----|----------------|-----------------|
| **Set aside dedicated preparation time** | Shows you take the opportunity seriously and gives you space to think through your answers. | Block a specific time slot (e.g., an hour the day before) and treat it like any other important meeting. |
| **Practice answers to common questions** | Rehearsing helps you articulate your thoughts clearly and reduces nerves. | Write out concise responses to typical questions (e.g., “Tell me about yourself,” “Why do you want this role?”) and say them out loud or with a friend. |
| **Understand the interview type and likely questions** | Different roles and industries focus on different competencies (technical, behavioral, case‑based, etc.). | Research the company and the specific job description; note any industry‑specific interview formats (e.g., technical screen, panel, 